# Lab 5 of 8 — Unsupervised Learning II: Anomaly Detection and Limited Labels
**TEB3123 Machine Learning** · Muhammad Aqil Bazli Bin Jafni

*Learn when labels are absent, rare, or expensive.* Coverage: Chapter 18; self-training and active learning previewed from Chapter 19.

**Goal.** Evaluate anomaly rankings with held-back labels; compare self-training and active learning under limited labels.

## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.semi_supervised import SelfTrainingClassifier

PROJECT_ROOT = Path.cwd().parent
DATA = PROJECT_ROOT / "all_datasets"

## Part A — Isolation Forest fraud ranking
Fit an Isolation Forest **without** `is_fraud` on 10,000 transactions (about 1.5% fraud). Points that are isolated by a few random splits get short paths and score as unusual. The held-back labels are used **only** to evaluate the ranking.

In [ ]:
fraud = pd.read_csv(DATA / "credit_card_fraud_dataset.csv")
print(fraud.head())
print("rows:", len(fraud), "  fraud rate:", round(fraud["is_fraud"].mean(), 4))

feature_cols = [
    "amount", "transaction_hour", "foreign_transaction", "location_mismatch",
    "device_trust_score", "velocity_last_24h", "cardholder_age",
]
X = fraud[feature_cols]
y = fraud["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

forest = IsolationForest(n_estimators=300, contamination="auto", random_state=42, n_jobs=-1)
forest.fit(X_train)                               # no labels passed
anomaly_score = -forest.decision_function(X_test)  # higher = more unusual

Report ROC AUC, average precision, and the operational number: precision among the top 100 alerts.

In [ ]:
prevalence = y_test.mean()
ap = average_precision_score(y_test, anomaly_score)
print("fraud prevalence:", round(prevalence, 4))
print("ROC AUC:", round(roc_auc_score(y_test, anomaly_score), 4))
print("average precision:", round(ap, 4), "(", round(ap / prevalence, 1), "x prevalence )")

ranked = X_test.copy()
ranked["fraud_label"] = y_test.to_numpy()
ranked["anomaly_score"] = anomaly_score
ranked = ranked.sort_values("anomaly_score", ascending=False)

review_budget = 100
top_k_precision = ranked.head(review_budget)["fraud_label"].mean()
print(f"fraud found in the top {review_budget}: {top_k_precision:.0%}")
print(ranked[["fraud_label", "anomaly_score", "amount"]].head(10))

Plot the score overlap and precision versus review budget.

In [ ]:
hits = ranked["fraud_label"].to_numpy()
k_values = np.arange(10, 1001, 10)
precision_at_k = [hits[:k].mean() for k in k_values]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].hist(anomaly_score[y_test.to_numpy() == 0], bins=30, alpha=0.65, label="non-fraud")
axes[0].hist(anomaly_score[y_test.to_numpy() == 1], bins=30, alpha=0.65, label="fraud")
axes[0].set_xlabel("Anomaly score")
axes[0].set_ylabel("Transactions")
axes[0].set_title("Isolation Forest score distribution")
axes[0].legend()

axes[1].plot(k_values, precision_at_k)
axes[1].axhline(prevalence, linestyle="--", color="grey", label=f"prevalence {prevalence:.1%}")
axes[1].set_xlabel("Number of top-ranked cases reviewed (k)")
axes[1].set_ylabel("Fraud rate among those k")
axes[1].set_title("Precision falls as the review budget grows")
axes[1].legend()
plt.tight_layout()
plt.show()

### Inspect — Isolation Forest
_To finalise once the real dataset has been run: ROC AUC, average precision, top-100 precision and what they mean for the review team._

### Try it yourself — review budgets of 50 and 300

In [ ]:
n_fraud_test = int(y_test.sum())
for budget in [50, 100, 300]:
    top = ranked.head(budget)["fraud_label"]
    print(f"budget {budget:3d}: precision = {top.mean():.1%}   "
          f"fraud caught = {int(top.sum()):2d} of {n_fraud_test} ({top.sum() / n_fraud_test:.0%} recall)   "
          f"lift = {top.mean() / prevalence:.1f}x prevalence")

_To finalise once the real dataset has been run: compare precision, recall and lift at budgets 50 and 300._

## Part B — Self-training and active learning with few labels
Split the breast-cancer rows into a **query pool** (where labels can be "bought"), a **validation set** for development decisions, and a **sealed test set** opened only once at the end.

In [ ]:
rng = np.random.default_rng(42)

cancer = pd.read_csv(DATA / "breast_cancer_dataset.csv")
X_c = cancer.drop(columns=["id", "diagnosis", "Unnamed: 32"], errors="ignore")
y_c = (cancer["diagnosis"] == "M").astype(int).to_numpy()   # 1 = malignant

X_dev, Xc_test, y_dev, yc_test = train_test_split(
    X_c, y_c, test_size=0.25, random_state=42, stratify=y_c
)
X_pool, X_val, y_pool, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, random_state=7, stratify=y_dev
)
X_pool = X_pool.reset_index(drop=True)
y_pool = np.asarray(y_pool)
print("pool:", len(X_pool), "  validation:", len(X_val), "  test:", len(Xc_test))

def make_model():
    return Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000)),
    ])

### Self-training
Hide 70% of the pool labels as −1. Self-training fits on the labelled rows, adds unlabelled rows whose predicted probability exceeds 0.90 as **pseudo-labels**, and refits. Wrong pseudo-labels can reinforce the model's own mistakes.

In [ ]:
y_partial = y_pool.copy()
y_partial[rng.random(len(y_partial)) < 0.70] = -1
print("labels kept:", int((y_partial != -1).sum()), "of", len(y_partial))

semi = Pipeline([
    ("scale", StandardScaler()),
    ("model", SelfTrainingClassifier(LogisticRegression(max_iter=2000), threshold=0.90)),
])
semi.fit(X_pool, y_partial)
print("self-training validation accuracy:",
      round(accuracy_score(y_val, semi.predict(X_val)), 3))

To judge whether self-training actually helped, compare it with a model trained on the **same labelled rows only**, and check how accurate the pseudo-labels were. (Validation set only — the test set stays sealed.)

In [ ]:
self_trainer = semi.named_steps["model"]
pseudo = self_trainer.labeled_iter_ > 0          # rows that started unlabelled and were pseudo-labelled
print("pseudo-labels added:", int(pseudo.sum()), "of", int((y_partial == -1).sum()), "hidden rows",
      " | self-training rounds:", self_trainer.n_iter_)
print("pseudo-label accuracy (checked against the hidden truth):",
      round((self_trainer.transduction_[pseudo] == y_pool[pseudo]).mean(), 3))

labelled = y_partial != -1
labelled_only = make_model().fit(X_pool[labelled], y_pool[labelled])
print("labelled rows only, validation accuracy:",
      round(accuracy_score(y_val, labelled_only.predict(X_val)), 3))

**Inspect.** Self-training reaches **0.963** validation accuracy, while a plain model on the same 89 labelled rows reaches **0.972**. Self-training added 201 pseudo-labels, and 99.5% of them were correct. It still didn't help, because the 0.90 threshold only accepts cases the model was already sure about. Those easy cases add little new information, and the one wrong pseudo-label pulls the boundary slightly the wrong way. With 89 labels on a well-separated dataset like this, the labelled data was already enough, so there was little room for self-training to add value.

### Active learning: uncertainty sampling vs random querying
Both policies start with the same 40 labels (20 per class) and buy 15 more per round. **Uncertainty sampling** queries the pool points whose predicted probability is closest to 0.5; **random querying** picks 15 at random. Same budget, same validation set.

In [ ]:
class0 = np.where(y_pool == 0)[0]
class1 = np.where(y_pool == 1)[0]
initial = np.concatenate([rng.choice(class0, 20, replace=False),
                          rng.choice(class1, 20, replace=False)])

active_labeled = set(initial.tolist())
random_labeled = set(initial.tolist())
rng_random = np.random.default_rng(123)
query_batch, rounds = 15, 8
label_counts, active_scores, random_scores = [], [], []

for round_no in range(rounds):
    active_idx, random_idx = sorted(active_labeled), sorted(random_labeled)
    active = make_model().fit(X_pool.iloc[active_idx], y_pool[active_idx])
    random_model = make_model().fit(X_pool.iloc[random_idx], y_pool[random_idx])

    label_counts.append(len(active_labeled))
    active_scores.append(accuracy_score(y_val, active.predict(X_val)))
    random_scores.append(accuracy_score(y_val, random_model.predict(X_val)))
    print(f"round {round_no + 1}: labels={label_counts[-1]:3d}  "
          f"uncertainty={active_scores[-1]:.3f}  random={random_scores[-1]:.3f}")

    if round_no == rounds - 1:
        break

    unlabeled = np.array([i for i in range(len(X_pool)) if i not in active_labeled])
    uncertainty = np.abs(active.predict_proba(X_pool.iloc[unlabeled])[:, 1] - 0.5)
    active_labeled.update(unlabeled[np.argsort(uncertainty)[:query_batch]].tolist())

    unlabeled = np.array([i for i in range(len(X_pool)) if i not in random_labeled])
    random_labeled.update(rng_random.choice(unlabeled, query_batch, replace=False).tolist())

plt.figure(figsize=(6, 3.6))
plt.plot(label_counts, active_scores, marker="o", label="uncertainty sampling")
plt.plot(label_counts, random_scores, marker="o", label="random querying")
plt.xlabel("Number of labelled pool examples")
plt.ylabel("Validation accuracy")
plt.title("Label efficiency during development")
plt.legend()
plt.tight_layout()
plt.show()

**Inspect.** Both curves flatten quickly, staying around 0.94–0.98 from 40 labels onwards. Uncertainty sampling shows **no clear gain** over random queries. It peaks higher once (0.981 at 70 labels), but it also dips lower (0.925 at 55), and it finishes slightly below random (0.953 vs 0.963). With only 107 validation rows, one misclassified case moves accuracy by about 0.009, so these gaps amount to one or two patients. That is noise, not a real difference. Starting with 40 balanced labels on a well-separated dataset leaves little uncertainty to exploit.

Both policies are finished, so the sealed test set is opened **once** for each.

In [ ]:
active_idx, random_idx = sorted(active_labeled), sorted(random_labeled)
active_final = make_model().fit(X_pool.iloc[active_idx], y_pool[active_idx])
random_final = make_model().fit(X_pool.iloc[random_idx], y_pool[random_idx])
print("final test accuracy, uncertainty sampling:",
      round(accuracy_score(yc_test, active_final.predict(Xc_test)), 3))
print("final test accuracy, random querying:     ",
      round(accuracy_score(yc_test, random_final.predict(Xc_test)), 3))

With 145 labels each, both policies score **0.972** on the sealed test set, the same result. Spending the budget on uncertain cases bought nothing here, compared with picking cases at random.

### Try it yourself — start with ten labels per class
The same experiment as a function, so it can be re-run with a smaller starting set. One run depends heavily on which points the random seed happens to pick, so I also repeat it over 20 seeds and average the curves. The validation set and the test set stay fixed, and the test set is still not used.

In [ ]:
def compare_policies(per_class, seed, rounds=8, query_batch=15, verbose=False):
    local_rng = np.random.default_rng(seed)
    start = np.concatenate([local_rng.choice(class0, per_class, replace=False),
                            local_rng.choice(class1, per_class, replace=False)])
    act, rnd = set(start.tolist()), set(start.tolist())
    rnd_rng = np.random.default_rng(seed + 1000)
    counts, act_acc, rnd_acc = [], [], []
    for r in range(rounds):
        a_idx, r_idx = sorted(act), sorted(rnd)
        a_model = make_model().fit(X_pool.iloc[a_idx], y_pool[a_idx])
        r_model = make_model().fit(X_pool.iloc[r_idx], y_pool[r_idx])
        counts.append(len(act))
        act_acc.append(accuracy_score(y_val, a_model.predict(X_val)))
        rnd_acc.append(accuracy_score(y_val, r_model.predict(X_val)))
        if verbose:
            print(f"round {r + 1}: labels={counts[-1]:3d}  uncertainty={act_acc[-1]:.3f}  random={rnd_acc[-1]:.3f}")
        if r == rounds - 1:
            break
        unl = np.array([i for i in range(len(X_pool)) if i not in act])
        unc = np.abs(a_model.predict_proba(X_pool.iloc[unl])[:, 1] - 0.5)
        act.update(unl[np.argsort(unc)[:query_batch]].tolist())
        unl = np.array([i for i in range(len(X_pool)) if i not in rnd])
        rnd.update(rnd_rng.choice(unl, query_batch, replace=False).tolist())
    return np.array(counts), np.array(act_acc), np.array(rnd_acc)

print("Single run, 10 labels per class (seed 42):")
_ = compare_policies(per_class=10, seed=42, verbose=True)

seeds = range(20)
summary = {}
for per_class in [10, 20]:
    runs = [compare_policies(per_class, s) for s in seeds]
    counts = runs[0][0]
    act_mean = np.mean([r[1] for r in runs], axis=0)
    rnd_mean = np.mean([r[2] for r in runs], axis=0)
    wins = np.mean([r[1] > r[2] for r in runs], axis=0)
    summary[per_class] = (counts, act_mean, rnd_mean)
    print(f"\nAveraged over {len(seeds)} seeds, {per_class} labels per class:")
    for n, a, b, w in zip(counts, act_mean, rnd_mean, wins):
        print(f"  labels={n:3d}  uncertainty={a:.3f}  random={b:.3f}  gap={a - b:+.3f}  "
              f"uncertainty strictly better in {w:.0%} of seeds")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, per_class in zip(axes, [10, 20]):
    counts, a, b = summary[per_class]
    ax.plot(counts, a, marker="o", label="uncertainty sampling")
    ax.plot(counts, b, marker="o", label="random querying")
    ax.set_title(f"Start: {per_class} labels per class (mean of 20 seeds)")
    ax.set_xlabel("Number of labelled pool examples")
axes[0].set_ylabel("Mean validation accuracy")
axes[0].legend()
plt.tight_layout()
plt.show()

**Reassessment.** A single run is too noisy to judge. With seed 42 and 10 labels per class, uncertainty sampling ends at 0.953 against 0.944 for random, but one run of the main experiment pointed the other way. Averaging over 20 seeds gives a clearer picture:

- **Starting from 10 per class**, uncertainty sampling gains a real early advantage. At 50 labels it averages **0.962 vs 0.943** (+0.019, better in 80% of seeds), and it reaches its plateau (~0.955) by about 50 labels. Random querying needs roughly 95–125 labels to catch up.
- **Starting from 20 per class**, the edge is smaller (about +0.01 at its peak), as in the original run.
- In both cases the gap **shrinks to almost nothing once about 100+ labels are bought**, because both models have enough data by then.

**Conclusion:** uncertainty sampling pays off when labels are genuinely scarce. A small starting set leaves the decision boundary uncertain, so choosing the ambiguous cases buys the most information per label. Once the model is reasonably good, the uncertain points are few and borderline, and random labels work just as well. The benefit is also small here because this dataset is easy (~95% accuracy from only 20–40 labels). On a harder problem, or with expensive labels such as a pathologist's review, saving roughly half the labels to reach the plateau would matter more.

## Verification checklist
- [x] The fraud labels are held back while the Isolation Forest is fitted (`forest.fit(X_train)` gets no `y`), and used only to evaluate it.
- [x] Precision at the actual review budget (top 100, plus 50 and 300) is reported alongside ROC AUC and average precision.
- [x] Self-training and active learning are compared on the same fixed validation and test sets (the `X_val` / `Xc_test` split made once at the start of Part B).
- [x] Uncertainty sampling is compared with random queries at equal label budgets, both from 20 per class and from 10 per class.

## Check your understanding (Chapter 18)

**1. Why is an anomaly score not automatically a probability of fraud, failure, or error?**

An anomaly score only measures how far a case departs from the learned reference pattern, for example how quickly an Isolation Forest isolates it. It is not calibrated against outcomes. The model never saw a fraud label, so it cannot know how often unusual cases turn out to be fraud. Many unusual cases are legitimate (a large one-off purchase abroad), and some fraud is designed to look normal. The scale is also arbitrary: scores from different models or settings don't map to the same likelihoods. In Part A, a top-ranked transaction is much more likely than average to be fraud, yet most top-ranked transactions are still not fraud. Turning a score into a probability would need labelled outcomes and calibration.

**2. How does a fixed review capacity turn anomaly detection into a ranking problem?**

If analysts can only investigate *k* cases per day, the real decision is not "is this an anomaly?" but "which *k* cases go to the front of the queue?" The threshold becomes "the top *k* scores", and only the order of the cases matters, not the absolute score values. So the right metric is **precision at k** (how many reviewed cases are real), plus recall at k (how much fraud is caught), rather than overall accuracy or a fixed cut-off such as |z| > 3. A model with a high ROC AUC can still waste the budget if its very top ranks are full of false alarms, which is why Part A reports precision at the actual budget.

**3. What is the difference between detecting outliers in the training sample and novelty detection for future observations?**

- **Outlier detection:** the training data is *contaminated*, meaning it already contains anomalies, and the goal is to find those anomalous rows in the data you have. The method has to be robust to the anomalies distorting the reference pattern (as outliers can inflate the mean and SD and mask themselves).
- **Novelty detection:** the training data is assumed to be *clean, normal* behaviour. The model learns that normal pattern and then flags **new, future** observations that deviate from it.

Part A is closer to outlier detection, because the training transactions include about 1.5% fraud that was never removed. It is evaluated like novelty detection, though: the model scores held-out transactions it never saw.